# 08c1 — Two-Layer Agent Locked Validation Preparation

## 这一步要验证什么

v2.2已经把“有直接证据时乱答”的风险降下来，但 Evidence-backed coverage 只有15%。

08c采用双层产品输出：

    第一层：Evidence-backed prediction
    有通过v2.2门控的直接证据时，输出支持或反对并展示证据。

    第二层：Model-based estimate
    没有直接证据时，使用只在Train训练的制度型模型给出估计，
    并明确标记这不是RAG直接证据。

    两层都不可靠：Insufficient evidence

排除全部旧题后，大选后只剩5个未见 division，因此本次锁定15个 division：
10个大选前高/中置信度对象，加上全部5个大选后对象。大选后2个低置信度对象会被单独标记。

08c1只完成全新样本锁定、免费检索、备用模型估计和费用预览。API调用为0，不使用Test。


## 为什么第二层不使用RAG语气

RAG检索到“相关背景”不等于检索到“该政党会支持或反对”的直接证据。

因此第二层不会生成虚假的引用，也不会说“根据证据，该党一定支持”。它只根据以下投票前可以知道的结构信息进行估计：

- 当前政党；
- 当前是执政党、主要反对党还是小反对党；
- 政策对象是否由政府背书；
- Motion family；
- Policy domain；
- 月份和少量议会程序特征。

备用模型不读取当前投票结果，也不读取Validation标签。


In [6]:
# 导入免费准备阶段需要的库；这里不导入OpenAI，也不会调用API
import hashlib
import json
import math
import re
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_columns", 180)

BASE_DIR = Path.cwd()
PROCESSED_DIR = BASE_DIR / "processed"
MODEL_DIR = PROCESSED_DIR / "model_v2"
RAG_DIR = PROCESSED_DIR / "rag_v3"
OBJECT_DIR = PROCESSED_DIR / "policy_object_v6"
FROZEN_RULE_DIR = PROCESSED_DIR / "locked_validation_failure_audit_v2_2"
OUTPUT_DIR = PROCESSED_DIR / "two_layer_validation_v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VALIDATION_PATH = MODEL_DIR / "model_validation_v2.csv"
TRAIN_PATH = MODEL_DIR / "model_train_v2.csv"
CHUNKS_PATH = RAG_DIR / "rag_chunks_v3.csv"
OBJECTS_PATH = OBJECT_DIR / "structured_policy_objects_v6.csv"
FROZEN_RULE_MANIFEST_PATH = FROZEN_RULE_DIR / "run_manifest_v2_2.json"

PARTIES = ["conservative", "green", "labour", "liberal-democrat"]
RANDOM_STATE = 20240921
ELECTION_DATE = pd.Timestamp("2024-07-05")
LOCKED_DIVISIONS = 15
QUERY_COUNT = LOCKED_DIVISIONS * len(PARTIES)

# 第二层模型参数在查看新标签前固定
FALLBACK_C = 0.5
FALLBACK_LOWER_THRESHOLD = 0.45
FALLBACK_UPPER_THRESHOLD = 0.55

# 08c2将继续使用同一小模型；08c1只预估费用
MODEL = "gpt-4o-mini-2024-07-18"
MAX_OUTPUT_TOKENS = 1200
INPUT_PRICE_PER_MILLION = 0.15
OUTPUT_PRICE_PER_MILLION = 0.60

print("Notebook version: 08c1-two-layer-locked-preparation-v1")
print("API calls: 0")
print("LLM calls: 0")
print("Final Test used: False")
print("Output directory:", OUTPUT_DIR)


Notebook version: 08c1-two-layer-locked-preparation-v1
API calls: 0
LLM calls: 0
Final Test used: False
Output directory: /Users/oooo1/01Course/2_70202Applying/reorganised/processed/two_layer_validation_v1


## 1. 读取Train、Validation、知识库与冻结规则

这里会检查v2.2是否真的被标记为可冻结。若冻结清单不存在或没有通过，08c1会停止。


In [7]:
# 读取数据，并建立统一的政策对象合同
def clean_text(value):
    """把缺失值和多余空格转换成安全字符串。"""
    if value is None or pd.isna(value):
        return ""
    return " ".join(str(value).split())


def first_text(*values):
    """返回第一个非空文本。"""
    for value in values:
        text = clean_text(value)
        if text:
            return text
    return ""


def native_bool(value):
    """把常见布尔值形式转成 Python bool。"""
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return clean_text(value).lower() in {"true", "1", "yes"}


def json_default(value):
    """把 NumPy 和 Pandas 类型转成 JSON 原生类型。"""
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    raise TypeError(f"Object of type {type(value).__name__} is not JSON serializable")


for path in [TRAIN_PATH, VALIDATION_PATH, CHUNKS_PATH, OBJECTS_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

validation = pd.read_csv(VALIDATION_PATH)
train = pd.read_csv(TRAIN_PATH)
chunks = pd.read_csv(CHUNKS_PATH)
objects = pd.read_csv(OBJECTS_PATH)

for frame in [train, validation, chunks, objects]:
    for column in frame.select_dtypes(include="object").columns:
        frame[column] = frame[column].fillna("")

validation["motion_date"] = pd.to_datetime(
    validation["motion_date"], errors="coerce"
)
chunks["source_date"] = pd.to_datetime(chunks["source_date"], errors="coerce")

object_columns = [
    "division_key", "vote_proposition", "canonical_policy_object",
    "policy_action", "source_quote", "source_quote_anchored",
    "extraction_confidence", "review_required", "review_priority",
    "final_policy_object", "final_motion_polarity", "issue_count_estimate",
]
object_contract = objects[object_columns].drop_duplicates("division_key").copy()
object_contract["contract_policy_object"] = object_contract.apply(
    lambda row: first_text(
        row["final_policy_object"],
        row["vote_proposition"],
        row["canonical_policy_object"],
    ),
    axis=1,
)
object_contract["contract_source_anchored"] = object_contract[
    "source_quote_anchored"
].map(native_bool)

print("Validation rows:", len(validation))
print("Validation divisions:", validation["division_key"].nunique())
print("Training rows for party-prior baseline:", len(train))
print("Knowledge-base chunks:", len(chunks))
print("Structured objects:", len(object_contract))


Validation rows: 570
Validation divisions: 169
Training rows for party-prior baseline: 2822
Knowledge-base chunks: 4942
Structured objects: 2091


In [8]:
# 验证08b v2.2冻结候选清单
if not FROZEN_RULE_MANIFEST_PATH.exists():
    raise FileNotFoundError(FROZEN_RULE_MANIFEST_PATH)

frozen_rule_manifest = json.loads(
    FROZEN_RULE_MANIFEST_PATH.read_text(encoding="utf-8")
)
if not frozen_rule_manifest.get("candidate_rules_frozen_for_08c", False):
    raise ValueError("v2.2尚未通过冻结门槛，不应创建08c样本。")
if frozen_rule_manifest.get("test_split_used") is not False:
    raise ValueError("冻结规则清单显示使用过Test，停止运行。")

frozen_rule_payload = json.dumps(
    frozen_rule_manifest,
    ensure_ascii=False,
    sort_keys=True,
).encode("utf-8")
FROZEN_RULE_MANIFEST_SHA256 = hashlib.sha256(
    frozen_rule_payload
).hexdigest()

print("Frozen rule version:", frozen_rule_manifest["notebook_version"])
print("Frozen rule manifest SHA-256:", FROZEN_RULE_MANIFEST_SHA256)


Frozen rule version: 08b-instrument-scope-patch-v2.2
Frozen rule manifest SHA-256: 597c2b2f4ca2869c0ae1c8d79ae05b1b06facc084dbe1777f55ee3de1c2f3a44


## 2. 排除所有曾参与开发或调试的 division

除了06–07阶段的案例，这里还会排除08使用过的20个 division。

这样08c面对的是全新议案，而不是再次考试已经看过的题目。


In [9]:
# 收集过往审计、人工检查和Agent评测使用过的 division
exclusion_files = [
    PROCESSED_DIR / "locked_validation_v1" / "locked_divisions_v1.csv",
    PROCESSED_DIR / "rag_audit_v2" / "audit_divisions_v2.csv",
    PROCESSED_DIR / "rag_hybrid_v7" / "evaluation_queries_v7.csv",
    PROCESSED_DIR / "policy_object_v6" / "temporal_audit_policy_objects_v6.csv",
    PROCESSED_DIR / "rag_agent_eval_v1" / "evaluation_cases_v1.csv",
    PROCESSED_DIR / "rag_agent_eval_v2" / "evaluation_cases_v2.csv",
    PROCESSED_DIR / "rag_agent_eval_v3" / "evaluation_cases_v3.csv",
]


def division_values_from_file(path):
    """从不同版本文件的候选列中提取 division_key。"""
    if not path.exists():
        return set()
    frame = pd.read_csv(path)
    candidate_columns = [
        "division_key", "query_division_key", "query_division",
    ]
    values = set()
    for column in candidate_columns:
        if column in frame.columns:
            values.update(
                clean_text(value)
                for value in frame[column].dropna().tolist()
                if clean_text(value)
            )
    return values


excluded_divisions = set()
exclusion_audit_rows = []
for path in exclusion_files:
    values = division_values_from_file(path)
    excluded_divisions.update(values)
    exclusion_audit_rows.append({
        "source_file": str(path),
        "file_exists": path.exists(),
        "excluded_divisions": len(values),
    })

exclusion_audit = pd.DataFrame(exclusion_audit_rows)
display(exclusion_audit)
print("Unique excluded divisions:", len(excluded_divisions))


,source_file,file_exists,excluded_divisions
0,/Users/oooo1/01Course/2_70202Applying/reorganised/processed/locked_validation_v1/locked_divisions_v1.csv,True,20
1,/Users/oooo1/01Course/2_70202Applying/reorganised/processed/rag_audit_v2/audit_divisions_v2.csv,True,24
2,/Users/oooo1/01Course/2_70202Applying/reorganised/processed/rag_hybrid_v7/evaluation_queries_v7.csv,True,30
3,/Users/oooo1/01Course/2_70202Applying/reorganised/processed/policy_object_v6/temporal_audit_policy_objects_v6.csv,True,22
4,/Users/oooo1/01Course/2_70202Applying/reorganised/processed/rag_agent_eval_v1/evaluation_cases_v1.csv,True,11
5,/Users/oooo1/01Course/2_70202Applying/reorganised/processed/rag_agent_eval_v2/evaluation_cases_v2.csv,True,11
6,/Users/oooo1/01Course/2_70202Applying/reorganised/processed/rag_agent_eval_v3/evaluation_cases_v3.csv,True,14


Unique excluded divisions: 74


## 3. 锁定新的15个 division，共60条政党查询

抽样规则在查看标签评分前固定：

- 10个大选前 division；
- 全部5个尚未见过的大选后 division；
- 尽量轮流覆盖不同政策领域；
- 每个 division 对四个政党各建立一条查询；
- 大选前只使用高/中置信度政策对象；
- 大选后允许最多2个有原文锚定的低置信度政策对象，并单独审计；
- 保存查询和标签 SHA-256，后续不允许重新抽样。


In [10]:
LOCKED_DIVISIONS_PATH = OUTPUT_DIR / "two_layer_divisions_v1.csv"
LOCKED_QUERIES_PATH = OUTPUT_DIR / "two_layer_queries_public_v1.csv"
LOCKED_LABELS_PATH = OUTPUT_DIR / "two_layer_labels_private_v1.csv"
LOCK_MANIFEST_PATH = OUTPUT_DIR / "two_layer_lock_manifest_v1.json"


def dataframe_sha256(frame, sort_columns):
    """对排序后的 CSV 内容计算稳定 SHA-256。"""
    ordered = frame.sort_values(sort_columns).reset_index(drop=True)
    payload = ordered.to_csv(index=False, lineterminator="\n")
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def round_robin_domain_sample(frame, n, seed):
    """在政策领域之间轮流抽样，并用固定随机种子处理领域内部顺序。"""
    working = frame.sample(frac=1, random_state=seed).copy()
    groups = {
        domain: group.reset_index(drop=True)
        for domain, group in working.groupby("policy_domain_primary", dropna=False)
    }
    domains = sorted(groups)
    selected_indices = []
    level = 0
    while len(selected_indices) < n:
        added = False
        for domain in domains:
            group = groups[domain]
            if level < len(group):
                selected_indices.append(group.iloc[level].name)
                # 保存原始 division_key，而不是临时整数索引
                selected_indices[-1] = group.iloc[level]["division_key"]
                added = True
                if len(selected_indices) == n:
                    break
        if not added:
            break
        level += 1
    selected = working[
        working["division_key"].isin(selected_indices)
    ].drop_duplicates("division_key")
    order = {key: i for i, key in enumerate(selected_indices)}
    selected["_sample_order"] = selected["division_key"].map(order)
    return selected.sort_values("_sample_order").drop(columns="_sample_order")


if all(path.exists() for path in [
    LOCKED_DIVISIONS_PATH, LOCKED_QUERIES_PATH,
    LOCKED_LABELS_PATH, LOCK_MANIFEST_PATH,
]):
    locked_divisions = pd.read_csv(LOCKED_DIVISIONS_PATH)
    locked_queries = pd.read_csv(LOCKED_QUERIES_PATH)
    locked_labels = pd.read_csv(LOCKED_LABELS_PATH)
    lock_manifest = json.loads(LOCK_MANIFEST_PATH.read_text(encoding="utf-8"))

    query_hash = dataframe_sha256(locked_queries, ["query_id"])
    label_hash = dataframe_sha256(locked_labels, ["query_id"])
    if query_hash != lock_manifest["locked_queries_sha256"]:
        raise ValueError("锁定查询 SHA-256 不匹配，文件可能被修改。")
    if label_hash != lock_manifest["locked_labels_sha256"]:
        raise ValueError("锁定标签 SHA-256 不匹配，文件可能被修改。")
    print("Loaded existing two-layer evaluation set.")
else:
    # division级字段只取一行，真实政党标签暂时分开保存
    division_fields = [
        "division_key", "motion_date", "motion_title_clean",
        "motion_text_clean", "legislation_name_clean", "motion_type",
        "motion_family", "policy_domain_primary",
    ]
    division_table = validation[division_fields].drop_duplicates(
        "division_key"
    ).merge(object_contract, on="division_key", how="left")

    label_quality = (
        validation.assign(
            valid_target=validation["target_policy_stance"].isin(
                ["support", "oppose"]
            )
        )
        .groupby("division_key")
        .agg(
            party_count=("party", "nunique"),
            valid_target_count=("valid_target", "sum"),
        )
        .reset_index()
    )
    division_table = division_table.merge(
        label_quality, on="division_key", how="left"
    )
    division_table["review_required_bool"] = division_table[
        "review_required"
    ].map(native_bool)
    division_table["object_eligible"] = (
        division_table["contract_policy_object"].map(clean_text).str.len().ge(8)
        & division_table["contract_source_anchored"].map(native_bool)
        & division_table["extraction_confidence"].isin(["high", "medium", "low"])
        & ~division_table["policy_action"].eq("multi_issue")
    )
    division_table["era"] = np.where(
        pd.to_datetime(division_table["motion_date"]) < ELECTION_DATE,
        "pre_2024_election",
        "post_2024_election",
    )

    eligible = division_table[
        division_table["object_eligible"]
        & division_table["party_count"].eq(4)
        & division_table["valid_target_count"].eq(4)
        & ~division_table["division_key"].isin(excluded_divisions)
    ].copy()

    pre_pool = eligible[
        eligible["era"].eq("pre_2024_election")
        & eligible["extraction_confidence"].isin(["high", "medium"])
    ]
    post_pool = eligible[eligible["era"].eq("post_2024_election")]
    pre = round_robin_domain_sample(pre_pool, 10, RANDOM_STATE)
    post = round_robin_domain_sample(post_pool, 5, RANDOM_STATE + 1)
    locked_divisions = pd.concat([pre, post], ignore_index=True)
    if len(locked_divisions) != LOCKED_DIVISIONS:
        raise ValueError(
            f"符合条件的锁定 division 不足：得到 {len(locked_divisions)} 个。"
        )

    label_columns = [
        "division_key", "party", "target_policy_stance",
        "target_binary_support",
    ]
    locked_labels = validation[label_columns].merge(
        locked_divisions[["division_key"]], on="division_key", how="inner"
    )
    locked_labels["query_id"] = (
        locked_labels["division_key"] + "__" + locked_labels["party"]
        + "__two_layer_v1"
    )
    locked_labels = locked_labels[[
        "query_id", "division_key", "party", "target_policy_stance",
        "target_binary_support",
    ]]

    public_columns = [
        "division_key", "motion_date", "motion_title_clean",
        "motion_text_clean", "legislation_name_clean", "motion_type",
        "motion_family", "policy_domain_primary", "era",
        "contract_policy_object", "policy_action", "source_quote",
        "extraction_confidence", "contract_source_anchored",
    ]
    locked_queries = locked_divisions[public_columns].merge(
        pd.DataFrame({"party": PARTIES}), how="cross"
    )
    locked_queries["query_id"] = (
        locked_queries["division_key"] + "__" + locked_queries["party"]
        + "__two_layer_v1"
    )

    locked_divisions.to_csv(LOCKED_DIVISIONS_PATH, index=False)
    locked_queries.to_csv(LOCKED_QUERIES_PATH, index=False)
    locked_labels.to_csv(LOCKED_LABELS_PATH, index=False)

    lock_manifest = {
        "lock_version": "two-layer-validation-v1",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "random_state": RANDOM_STATE,
        "locked_divisions": int(len(locked_divisions)),
        "locked_queries": int(len(locked_queries)),
        "excluded_divisions": int(len(excluded_divisions)),
        "locked_queries_sha256": dataframe_sha256(
            locked_queries, ["query_id"]
        ),
        "locked_labels_sha256": dataframe_sha256(
            locked_labels, ["query_id"]
        ),
        "test_split_used": False,
    }
    LOCK_MANIFEST_PATH.write_text(
        json.dumps(lock_manifest, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print("Created new two-layer evaluation set.")

display(locked_divisions[[
    "division_key", "motion_date", "era", "motion_family",
    "policy_domain_primary", "contract_policy_object",
]])
print("Locked queries SHA-256:", lock_manifest["locked_queries_sha256"])
print("Locked labels SHA-256:", lock_manifest["locked_labels_sha256"])


Created new two-layer evaluation set.


,division_key,motion_date,era,motion_family,policy_domain_primary,contract_policy_object
0,pw-2024-04-30-144-commons,2024-04-30,pre_2024_election,amendment_or_clause,digital_data_media_ai,disagrees with Lords amendment 31
1,pw-2024-02-20-76-commons,2024-02-20,pre_2024_election,amendment_or_clause,economy_finance_business,"Proposed amendment to Clause 1 - Duty to invite applications for offshore licences: Amendment proposed: 17, page 1, line 6, at end insert (aa) the climate change test (see sect..."
2,pw-2024-02-27-83-commons,2024-02-27,pre_2024_election,amendment_or_clause,housing_planning_local_government,Proposed amendment to Amendment: — — nsolvency of certain persons with an interest in higher-risk and relevant buildings: Schedule 2 - Determining and sharing the market value:...
3,pw-2024-04-17-125-commons,2024-04-17,pre_2024_election,amendment_or_clause,immigration_asylum,disagrees with Lords amendment 3E
4,pw-2024-03-13-93-commons,2024-03-13,pre_2024_election,amendment_or_clause,other_or_unclear,Proposed New Clause: New Clause 2: National Insurance Contributions (Reduction in Rates) (No. 2) Bill — New Clause 2 - Review of effects of frozen thresholds he Treasury must l...
5,pw-2024-04-15-119-commons,2024-04-15,pre_2024_election,amendment_or_clause,economy_finance_business,disagrees with the Lords in their Amendment 6B
6,pw-2024-05-22-157-commons,2024-05-22,pre_2024_election,bill_stage,immigration_asylum,"Proposed Bill: Leave for Bill: Immigration and Asylum — I beg to move, That leave be given to bring in a Bill to make provision for the annual approval by the House of Commons ..."
7,pw-2024-04-15-122-commons,2024-04-15,pre_2024_election,amendment_or_clause,other_or_unclear,disagrees with the Lords in their amendment 10B
8,pw-2024-03-19-109-commons,2024-03-19,pre_2024_election,amendment_or_clause,economy_finance_business,Proposed New Clause: New Clause 5: Trade (Comprehensive and Progressive Agreement for Trans-Pacific Partnership) Bill [Lords] — New Clause 5 - Review: Investor-State Dispute Se...
9,pw-2024-03-18-105-commons,2024-03-18,pre_2024_election,amendment_or_clause,immigration_asylum,disagrees with Lords amendment 8


Locked queries SHA-256: 90acf73e828b73bc0e3d54794f3a18b1e8fe7614f6434625f9961a021d47c1cc
Locked labels SHA-256: 73a899c20e521bc578661c2a66cfc87d33ce284ad8fc260423ab01026d83c51b


## 4. 免费、严格时间滚动的RAG检索

检索只允许使用查询日期之前的资料，并排除当前 division 自己。

08c继续使用已经审计过的 TF-IDF rolling index，不在新样本上更换 Dense 权重或检索参数。


In [11]:
# 定义和06e一致的文本规范化与政策锚点方法
SYNONYM_GROUPS = [
    (r"\b(?:value added tax|vat)\b", " vat value added tax "),
    (r"\b(?:private schools?|independent schools?)\b", " private school independent school "),
    (r"\b(?:school fees?|fees paid to schools?)\b", " school fee tuition fee "),
    (r"\b(?:two child limit|two-child limit|two child cap|two-child cap)\b", " two child limit two child cap "),
    (r"\b(?:general practitioners?|gps?)\b", " gp general practitioner primary care doctor "),
    (r"\b(?:national insurance contributions?|nics?)\b", " national insurance employer contribution nic "),
    (r"\b(?:railways?|railroads?)\b", " railway rail public transport "),
    (r"\b(?:vapes?|vaping)\b", " vape vaping electronic cigarette "),
    (r"\b(?:tobacco|cigarettes?|smoking)\b", " tobacco cigarette smoking "),
]

PROCEDURAL_PATTERNS = [
    r"\bfirst reading\b", r"\bsecond reading\b", r"\bthird reading\b",
    r"\bcommencement\b", r"\bquestion put\b", r"\bstanding order\b",
    r"\bthat the bill be now read\b", r"\bthat the clause stand part\b",
    r"\bnew clause\s*\d*\b", r"\bclause\s*\d+\b",
    r"\bamendment proposed\b", r"\bamendment\b",
    r"\bi beg to move\b", r"\bthe house\b",
]

CONTENT_STOP_WORDS = set(ENGLISH_STOP_WORDS).union({
    "bill", "act", "order", "motion", "reading", "clause", "amendment",
    "question", "house", "commons", "approve", "proposed", "stage",
    "page", "line", "section", "part", "paragraph", "schedule",
})


def normalize_retrieval_text(value):
    """统一同义词并删除只表示议会程序的固定短语。"""
    text = clean_text(value).lower().replace("__", " ")
    for pattern in PROCEDURAL_PATTERNS:
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE)
    for pattern, replacement in SYNONYM_GROUPS:
        text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
    text = re.sub(r"[^a-z0-9£%\- ]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def content_tokens(value):
    """提取真正表示政策主题的词。"""
    return {
        token
        for token in re.findall(r"[a-z][a-z\-]{2,}", normalize_retrieval_text(value))
        if token not in CONTENT_STOP_WORDS
    }


def token_coverage(query_value, evidence_value):
    """计算查询政策词被证据覆盖的比例。"""
    query_tokens = content_tokens(query_value)
    evidence_tokens = content_tokens(evidence_value)
    if not query_tokens or not evidence_tokens:
        return 0.0
    return len(query_tokens & evidence_tokens) / len(query_tokens)


chunks["retrieval_text_v4"] = (
    chunks["title"].map(normalize_retrieval_text)
    + " " + chunks["retrieval_text"].map(normalize_retrieval_text)
).str.strip()


def make_vectorizer():
    """创建词和双词短语 TF-IDF。"""
    return TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=1,
        max_df=0.98,
        sublinear_tf=True,
        norm="l2",
        max_features=100000,
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z\-]{1,}\b",
    )


rolling_indexes = {}
for query_date in sorted(pd.to_datetime(locked_queries["motion_date"]).unique()):
    query_date = pd.Timestamp(query_date)
    available_mask = chunks["source_date"].notna() & chunks["source_date"].lt(query_date)
    available_indices = np.flatnonzero(available_mask.to_numpy())
    vectorizer = make_vectorizer()
    matrix = vectorizer.fit_transform(
        chunks.iloc[available_indices]["retrieval_text_v4"]
    )
    rolling_indexes[query_date] = {
        "indices": available_indices,
        "vectorizer": vectorizer,
        "matrix": matrix,
    }

rolling_audit = pd.DataFrame([
    {
        "query_date": date,
        "available_chunks": len(value["indices"]),
        "vocabulary_size": len(value["vectorizer"].vocabulary_),
    }
    for date, value in rolling_indexes.items()
])
display(rolling_audit)


,query_date,available_chunks,vocabulary_size
0,2024-02-20,3152,21708
1,2024-02-27,3164,21811
2,2024-03-13,3189,22157
3,2024-03-18,3205,22296
4,2024-03-19,3245,22298
5,2024-04-15,3273,22502
6,2024-04-17,3296,22542
7,2024-04-30,3345,22732
8,2024-05-22,3404,23424
9,2024-11-06,4690,65319


In [12]:
# 沿用06e的来源门槛和最大证据数量
MAX_RESULTS = 8
MAX_POLICY_RESULTS = 2
MAX_HISTORICAL_RESULTS = 4
MAX_BILL_RESULTS = 1
MAX_CHUNKS_PER_DOCUMENT = 2

POLICY_ABS_MIN = 0.030
POLICY_RELATIVE_TO_TOP = 0.55
POLICY_ANCHOR_MIN = 0.08
POLICY_STRONG_BASE = 0.090
HISTORICAL_BASE_MIN = 0.050
HISTORICAL_RELATIVE_TO_TOP = 0.60
HISTORICAL_ANCHOR_MIN = 0.10
HISTORICAL_STRONG_BASE = 0.105
BILL_ABS_MIN = 0.080
BILL_ANCHOR_MIN = 0.12
DOMAIN_MATCH_BOOST = 0.012
ANCHOR_COVERAGE_WEIGHT = 0.120
DIRECT_MATCH_BOOST = 0.180


def build_query_text(row):
    """提高政策对象权重，同时保留标题、法案名和部分正文。"""
    title = normalize_retrieval_text(row["motion_title_clean"])
    policy_object = normalize_retrieval_text(row["contract_policy_object"])
    legislation = normalize_retrieval_text(row["legislation_name_clean"])
    motion_text = normalize_retrieval_text(clean_text(row["motion_text_clean"])[:1800])
    domain = normalize_retrieval_text(row["policy_domain_primary"].replace("_", " "))
    return " ".join([
        title, title, title,
        policy_object, policy_object, policy_object, policy_object,
        legislation, legislation, domain, motion_text,
    ]).strip()


def build_query_anchor(row):
    """锚点不使用冗长正文。"""
    return " ".join([
        clean_text(row["motion_title_clean"]),
        clean_text(row["contract_policy_object"]),
        clean_text(row["legislation_name_clean"]),
    ])


def direct_policy_match(row, evidence_title):
    """判断证据标题是否直接包含当前政策或法案名称。"""
    evidence_value = normalize_retrieval_text(evidence_title)
    if not evidence_value:
        return False
    query_values = [
        normalize_retrieval_text(row["contract_policy_object"]),
        normalize_retrieval_text(row["legislation_name_clean"]),
        normalize_retrieval_text(row["motion_title_clean"]),
    ]
    for query_value in query_values:
        if not query_value:
            continue
        shorter = min(
            len(content_tokens(query_value)), len(content_tokens(evidence_value))
        )
        if shorter >= 2 and (
            query_value in evidence_value or evidence_value in query_value
        ):
            return True
    return False


def score_candidates(query_row):
    """只在查询日期之前、当前政党或公共来源中评分。"""
    query_date = pd.Timestamp(query_row["motion_date"])
    rolling = rolling_indexes[query_date]
    available = chunks.iloc[rolling["indices"]].copy()
    party_ok = available["party"].isin([query_row["party"], "all"])
    division_ok = available["division_key"].ne(
        query_row["division_key"]
    ).fillna(True)
    local_positions = np.flatnonzero((party_ok & division_ok).to_numpy())
    if not len(local_positions):
        return pd.DataFrame()

    query_vector = rolling["vectorizer"].transform([
        build_query_text(query_row)
    ])
    all_scores = (rolling["matrix"] @ query_vector.T).toarray().ravel()
    candidates = available.iloc[local_positions].copy()
    candidates["base_score"] = all_scores[local_positions]
    anchor = build_query_anchor(query_row)
    candidates["anchor_coverage"] = candidates.apply(
        lambda row: token_coverage(
            anchor,
            clean_text(row["title"]) + " "
            + clean_text(row["retrieval_text_v4"])[:1200],
        ),
        axis=1,
    )
    candidates["direct_policy_match"] = candidates["title"].map(
        lambda title: direct_policy_match(query_row, title)
    )
    candidates["same_domain"] = candidates["policy_domain"].eq(
        clean_text(query_row["policy_domain_primary"])
    )
    candidates["domain_boost"] = np.where(
        candidates["source_type"].eq("historical_vote")
        & candidates["same_domain"],
        DOMAIN_MATCH_BOOST,
        0.0,
    )
    candidates["adjusted_score"] = (
        candidates["base_score"]
        + candidates["anchor_coverage"] * ANCHOR_COVERAGE_WEIGHT
        + candidates["direct_policy_match"].astype(float) * DIRECT_MATCH_BOOST
        + candidates["domain_boost"]
    )
    return candidates.sort_values(
        ["adjusted_score", "base_score"], ascending=False
    )


def add_selected(candidate, selected, document_counts, channel):
    """限制重复 chunk 和单一文档占用全部位置。"""
    if any(item["chunk_id"] == candidate["chunk_id"] for item in selected):
        return False
    document_id = candidate["document_id"]
    if document_counts.get(document_id, 0) >= MAX_CHUNKS_PER_DOCUMENT:
        return False
    item = candidate.to_dict()
    item["evidence_channel"] = channel
    selected.append(item)
    document_counts[document_id] = document_counts.get(document_id, 0) + 1
    return True


def select_semantic_evidence(candidates):
    """应用来源专属门槛，最多返回8条证据。"""
    if candidates.empty:
        return []
    selected = []
    document_counts = {}

    policy = candidates[candidates["source_type"].eq("manifesto")].copy()
    if len(policy):
        threshold = max(POLICY_ABS_MIN, policy["base_score"].max() * POLICY_RELATIVE_TO_TOP)
        policy = policy[
            policy["base_score"].ge(threshold)
            & (
                policy["anchor_coverage"].ge(POLICY_ANCHOR_MIN)
                | policy["base_score"].ge(POLICY_STRONG_BASE)
                | policy["direct_policy_match"]
            )
        ]
        for _, candidate in policy.head(MAX_POLICY_RESULTS).iterrows():
            add_selected(candidate, selected, document_counts, "policy")

    historical = candidates[candidates["source_type"].eq("historical_vote")].copy()
    if len(historical):
        relative = historical["adjusted_score"].max() * HISTORICAL_RELATIVE_TO_TOP
        historical = historical[
            historical["adjusted_score"].ge(relative)
            & historical["base_score"].ge(HISTORICAL_BASE_MIN)
            & (
                historical["anchor_coverage"].ge(HISTORICAL_ANCHOR_MIN)
                | (
                    historical["base_score"].ge(HISTORICAL_STRONG_BASE)
                    & historical["anchor_coverage"].ge(0.04)
                )
                | historical["direct_policy_match"]
            )
        ]
        for _, candidate in historical.head(MAX_HISTORICAL_RESULTS).iterrows():
            add_selected(candidate, selected, document_counts, "historical")

    bills = candidates[candidates["source_type"].eq("bill_reference")].copy()
    bills = bills[
        bills["direct_policy_match"]
        | (
            bills["base_score"].ge(BILL_ABS_MIN)
            & bills["anchor_coverage"].ge(BILL_ANCHOR_MIN)
        )
    ]
    for _, candidate in bills.head(MAX_BILL_RESULTS).iterrows():
        add_selected(candidate, selected, document_counts, "bill")

    selected = sorted(
        selected,
        key=lambda item: (item["adjusted_score"], item["base_score"]),
        reverse=True,
    )
    return selected[:MAX_RESULTS]


In [13]:
# 对80条锁定查询运行免费检索
retrieval_rows = []
query_rows = []

for _, query in locked_queries.sort_values("query_id").iterrows():
    candidates = score_candidates(query)
    selected = select_semantic_evidence(candidates)

    for rank, item in enumerate(selected, start=1):
        retrieval_rows.append({
            "query_id": query["query_id"],
            "query_division_key": query["division_key"],
            "query_party": query["party"],
            "query_date": query["motion_date"],
            "query_title": query["motion_title_clean"],
            "query_policy_object": query["contract_policy_object"],
            "query_parliamentary_procedure": query["policy_action"],
            "query_motion_excerpt": clean_text(query["motion_text_clean"])[:1800],
            "query_domain": query["policy_domain_primary"],
            "query_era": query["era"],
            "evidence_id": f"E{rank}",
            "retrieval_rank": rank,
            "source_type": item["source_type"],
            "evidence_channel": item["evidence_channel"],
            "base_score": item["base_score"],
            "adjusted_score": item["adjusted_score"],
            "anchor_coverage": item["anchor_coverage"],
            "direct_policy_match": item["direct_policy_match"],
            "same_domain": item["same_domain"],
            "evidence_chunk_id": item["chunk_id"],
            "evidence_document_id": item["document_id"],
            "evidence_title": item["title"],
            "evidence_party": item["party"],
            "evidence_date": item["source_date"],
            "evidence_division_key": item["division_key"],
            "evidence_policy_domain": item["policy_domain"],
            "stance_label": item["stance_label"],
            "source_url": item["source_url"],
            "page_number": item["page_number"],
            "evidence_text": item["evidence_text"],
        })

    query_rows.append({
        "query_id": query["query_id"],
        "division_key": query["division_key"],
        "party": query["party"],
        "query_date": query["motion_date"],
        "motion_title": query["motion_title_clean"],
        "query_policy_object": query["contract_policy_object"],
        "query_parliamentary_procedure": query["policy_action"],
        "motion_excerpt": clean_text(query["motion_text_clean"])[:1800],
        "policy_domain_primary": query["policy_domain_primary"],
        "motion_family": query["motion_family"],
        "era": query["era"],
        "retrieved_evidence": len(selected),
    })

retrieval = pd.DataFrame(retrieval_rows)
queries = pd.DataFrame(query_rows)

# 给历史证据补上历史政策对象和归一化立场
historical_contract = object_contract.add_prefix("historical_").rename(
    columns={"historical_division_key": "evidence_division_key"}
)
if not retrieval.empty:
    retrieval = retrieval.merge(
        historical_contract, on="evidence_division_key", how="left"
    )
    retrieval["historical_substantive_policy_object"] = retrieval[
        "historical_contract_policy_object"
    ].map(clean_text)
    retrieval["historical_parliamentary_procedure"] = retrieval[
        "historical_policy_action"
    ].map(clean_text)
    retrieval["historical_stance_toward_object"] = np.where(
        retrieval["source_type"].eq("historical_vote"),
        retrieval["stance_label"].map(clean_text),
        "",
    )
    retrieval["historical_contract_complete"] = np.where(
        retrieval["source_type"].eq("historical_vote"),
        retrieval["historical_substantive_policy_object"].str.len().gt(0)
        & retrieval["historical_stance_toward_object"].isin(["support", "oppose"]),
        True,
    )

queries.to_csv(OUTPUT_DIR / "two_layer_query_runtime_v1.csv", index=False)
retrieval.to_csv(OUTPUT_DIR / "two_layer_retrieval_evidence_v1.csv", index=False)

display(queries["retrieved_evidence"].value_counts().sort_index().to_frame("queries"))
if not retrieval.empty:
    display(retrieval["source_type"].value_counts().to_frame("evidence_rows"))


,queries
retrieved_evidence,
1,4
4,34
5,7
6,10
7,5


,evidence_rows
source_type,
historical_vote,213
manifesto,33
bill_reference,24


## 5. 免费结构、泄漏和样本锁定检查

任何结构门失败都会停止运行。特别检查：

- 新样本与全部开发样本没有 division 重叠；
- 所有证据严格早于查询日期；
- 没有检索当前 division；
- 没有重复 chunk；
- 每个政党恰好20条；
- Test完全未使用。


In [14]:
# 计算结构检查
query_hash_now = dataframe_sha256(locked_queries, ["query_id"])
label_hash_now = dataframe_sha256(locked_labels, ["query_id"])
overlap_with_debug = set(locked_divisions["division_key"]) & excluded_divisions

if retrieval.empty:
    temporal_leakage_count = 0
    same_division_count = 0
    duplicate_chunk_count = 0
    max_results = 0
else:
    temporal_leakage_count = int((
        pd.to_datetime(retrieval["evidence_date"], errors="coerce")
        >= pd.to_datetime(retrieval["query_date"], errors="coerce")
    ).sum())
    same_division_count = int((
        retrieval["evidence_division_key"].map(clean_text).ne("")
        & retrieval["evidence_division_key"].eq(retrieval["query_division_key"])
    ).sum())
    duplicate_chunk_count = int(
        retrieval.duplicated(["query_id", "evidence_chunk_id"]).sum()
    )
    max_results = int(retrieval.groupby("query_id").size().max())

STRUCTURAL_GATES = {
    "twenty_divisions_gate": locked_divisions["division_key"].nunique() == LOCKED_DIVISIONS,
    "eighty_queries_gate": len(locked_queries) == QUERY_COUNT,
    "twenty_queries_per_party_gate": (
        locked_queries["party"].value_counts().eq(LOCKED_DIVISIONS).all()
    ),
    "pre_post_balance_gate": (
        locked_divisions["era"].value_counts().reindex(
            ["pre_2024_election", "post_2024_election"], fill_value=0
        ).eq(pd.Series({
            "pre_2024_election": 10,
            "post_2024_election": 5,
        })).all()
    ),
    "debug_overlap_gate": len(overlap_with_debug) == 0,
    "query_hash_gate": query_hash_now == lock_manifest["locked_queries_sha256"],
    "label_hash_gate": label_hash_now == lock_manifest["locked_labels_sha256"],
    "query_object_gate": locked_queries["contract_policy_object"].map(
        clean_text
    ).str.len().ge(8).all(),
    "temporal_leakage_gate": temporal_leakage_count == 0,
    "same_division_gate": same_division_count == 0,
    "duplicate_chunk_gate": duplicate_chunk_count == 0,
    "maximum_eight_results_gate": max_results <= 8,
    "low_confidence_cap_gate": int(
        locked_divisions["extraction_confidence"].eq("low").sum()
    ) <= 2,
    "low_confidence_source_anchored_gate": locked_divisions.loc[
        locked_divisions["extraction_confidence"].eq("low"),
        "contract_source_anchored",
    ].map(native_bool).all(),
    "test_split_gate": True,
}

print("Structural gates:", STRUCTURAL_GATES)
print("Policy domains:", locked_divisions["policy_domain_primary"].nunique())
print("Motion families:", locked_divisions["motion_family"].nunique())
print("Mean evidence per query:", round(queries["retrieved_evidence"].mean(), 3))
print("Queries with no evidence:", int(queries["retrieved_evidence"].eq(0).sum()))

if not all(bool(value) for value in STRUCTURAL_GATES.values()):
    raise ValueError("锁定评测结构检查未全部通过，不应开启付费调用。")


Structural gates: {'twenty_divisions_gate': True, 'eighty_queries_gate': True, 'twenty_queries_per_party_gate': True, 'pre_post_balance_gate': True, 'debug_overlap_gate': True, 'query_hash_gate': True, 'label_hash_gate': True, 'query_object_gate': True, 'temporal_leakage_gate': True, 'same_division_gate': True, 'duplicate_chunk_gate': True, 'maximum_eight_results_gate': True, 'low_confidence_cap_gate': True, 'low_confidence_source_anchored_gate': True, 'test_split_gate': True}
Policy domains: 5
Motion families: 2
Mean evidence per query: 4.5
Queries with no evidence: 0


## 6. 训练第二层制度型估计模型

这不是RAG，也不提供引用。它只是一个 Logistic Regression：根据制度与议案结构输出支持分数。

为什么选择 Logistic Regression：

- 目标是支持/反对二分类；
- 它可以输出0到1之间的支持分数；
- 比复杂模型更容易解释；
- 类别特征经过 One-hot encoding 后，每个政党、角色和政策领域都有独立权重。

`C=0.5` 使用之前04c的固定正则化强度，不根据08c标签重新选择。0.45–0.55之间视为过于接近五五开，第二层也拒答。


In [15]:
# 构造投票前可获得的制度特征
def government_backing_status(frame):
    """把政府背书信息转换成三个可解释类别。"""
    known = pd.to_numeric(
        frame["government_backing_known"], errors="coerce"
    ).fillna(0).astype(int)
    backed = frame["final_object_government_backed"].map(native_bool)
    return np.where(
        known.eq(0),
        "unknown",
        np.where(backed, "government_backed", "not_government_backed"),
    )


train_fallback = train[
    train["target_binary_support"].isin([0, 1])
].copy()

# 从Validation表取新锁定查询的结构特征，但不取标签
fallback_input_columns = [
    "division_key", "party", "party_role", "motion_family",
    "policy_domain_primary", "government_backing_known",
    "final_object_government_backed", "motion_month",
    "is_governing_party", "is_main_opposition", "is_opposition_day",
    "is_amendment", "is_new_clause", "is_financial_motion",
]
fallback_input = locked_queries[[
    "query_id", "division_key", "party",
]].merge(
    validation[fallback_input_columns].drop_duplicates(
        ["division_key", "party"]
    ),
    on=["division_key", "party"],
    how="left",
    validate="one_to_one",
)

train_fallback["government_backing_status"] = government_backing_status(
    train_fallback
)
fallback_input["government_backing_status"] = government_backing_status(
    fallback_input
)

for frame in [train_fallback, fallback_input]:
    frame["role_backing_interaction"] = (
        frame["party_role"].map(clean_text)
        + "__"
        + frame["government_backing_status"].map(clean_text)
    )

CATEGORICAL_FEATURES = [
    "party", "party_role", "government_backing_status",
    "role_backing_interaction", "motion_family", "policy_domain_primary",
]
NUMERIC_FEATURES = [
    "motion_month", "is_governing_party", "is_main_opposition",
    "is_opposition_day", "is_amendment", "is_new_clause",
    "is_financial_motion",
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            CATEGORICAL_FEATURES,
        ),
        (
            "numeric",
            StandardScaler(with_mean=False),
            NUMERIC_FEATURES,
        ),
    ],
    remainder="drop",
)

fallback_model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
        LogisticRegression(
            C=FALLBACK_C,
            class_weight="balanced",
            solver="liblinear",
            max_iter=3000,
            random_state=RANDOM_STATE,
        ),
    ),
])

fallback_model.fit(
    train_fallback[CATEGORICAL_FEATURES + NUMERIC_FEATURES],
    train_fallback["target_binary_support"].astype(int),
)

fallback_input["model_support_score"] = fallback_model.predict_proba(
    fallback_input[CATEGORICAL_FEATURES + NUMERIC_FEATURES]
)[:, 1]
fallback_input["model_estimated_stance"] = np.select(
    [
        fallback_input["model_support_score"].ge(
            FALLBACK_UPPER_THRESHOLD
        ),
        fallback_input["model_support_score"].le(
            FALLBACK_LOWER_THRESHOLD
        ),
    ],
    ["support", "oppose"],
    default="insufficient_evidence",
)
fallback_input["output_tier_if_rag_abstains"] = np.where(
    fallback_input["model_estimated_stance"].isin(["support", "oppose"]),
    "model_based_estimate",
    "insufficient_evidence",
)

# Party prior只作为比较基线，不作为产品最终解释
party_prior = (
    train_fallback.groupby("party")["target_binary_support"]
    .mean()
    .rename("train_party_support_rate")
)
fallback_input = fallback_input.merge(
    party_prior,
    left_on="party",
    right_index=True,
    how="left",
    validate="many_to_one",
)

fallback_public_columns = [
    "query_id", "division_key", "party", "party_role",
    "government_backing_status", "motion_family",
    "policy_domain_primary", "model_support_score",
    "model_estimated_stance", "output_tier_if_rag_abstains",
    "train_party_support_rate",
]
fallback_estimates = fallback_input[fallback_public_columns].copy()
fallback_estimates.to_csv(
    OUTPUT_DIR / "fallback_estimates_public_v1.csv", index=False
)

print("Fallback estimate distribution")
display(
    fallback_estimates["model_estimated_stance"]
    .value_counts(dropna=False)
    .to_frame("queries")
)
print("Fallback support-score summary")
display(fallback_estimates["model_support_score"].describe().to_frame())


Fallback estimate distribution


,queries
model_estimated_stance,
support,25
oppose,21
insufficient_evidence,14


Fallback support-score summary


,model_support_score
count,60.000000
mean,0.518668
std,0.141349
min,0.219871
25%,0.424876
50%,0.524054
75%,0.591966
max,0.830897


## 7. 费用预览与冻结清单

费用预览使用字符数近似Token数，只用于设置上限。真正费用会在08c2读取API返回的Token usage重新计算。

这一步不会请求API Key，也不会执行任何付费调用。


In [16]:
# 根据查询和检索证据长度估计最多60条调用的费用
if retrieval.empty:
    evidence_chars_by_query = pd.Series(dtype=float)
else:
    evidence_chars_by_query = retrieval.groupby("query_id")[
        "evidence_text"
    ].apply(lambda values: sum(len(clean_text(value)) for value in values))

cost_preview = queries[["query_id", "motion_excerpt"]].copy()
cost_preview["query_chars"] = (
    cost_preview["motion_excerpt"].map(clean_text).str.len()
)
cost_preview["evidence_chars"] = (
    cost_preview["query_id"].map(evidence_chars_by_query).fillna(0)
)
cost_preview["estimated_input_tokens"] = (
    # 额外900 tokens近似系统Prompt和结构化元数据
    (cost_preview["query_chars"] + cost_preview["evidence_chars"]) / 4
    + 900
).round().astype(int)
cost_preview["maximum_output_tokens"] = MAX_OUTPUT_TOKENS
cost_preview["estimated_max_cost_usd"] = (
    cost_preview["estimated_input_tokens"]
    * INPUT_PRICE_PER_MILLION / 1_000_000
    + cost_preview["maximum_output_tokens"]
    * OUTPUT_PRICE_PER_MILLION / 1_000_000
)

estimated_max_cost = float(cost_preview["estimated_max_cost_usd"].sum())
cost_preview.to_csv(OUTPUT_DIR / "api_cost_preview_v1.csv", index=False)

preparation_manifest = {
    "notebook_version": "08c1-two-layer-locked-preparation-v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "random_state": RANDOM_STATE,
    "frozen_rule_version": frozen_rule_manifest["notebook_version"],
    "frozen_rule_manifest_sha256": FROZEN_RULE_MANIFEST_SHA256,
    "locked_queries_sha256": lock_manifest["locked_queries_sha256"],
    "locked_labels_sha256": lock_manifest["locked_labels_sha256"],
    "fallback_model": "structured_logistic_regression",
    "fallback_C": FALLBACK_C,
    "fallback_lower_threshold": FALLBACK_LOWER_THRESHOLD,
    "fallback_upper_threshold": FALLBACK_UPPER_THRESHOLD,
    "api_calls": 0,
    "llm_calls": 0,
    "test_split_used": False,
    "structural_gates": STRUCTURAL_GATES,
    "estimated_max_api_cost_usd": estimated_max_cost,
}

with (OUTPUT_DIR / "preparation_manifest_v1.json").open(
    "w", encoding="utf-8"
) as handle:
    json.dump(
        preparation_manifest,
        handle,
        ensure_ascii=False,
        indent=2,
        default=lambda value: value.item()
        if isinstance(value, np.generic) else str(value),
    )

print("Estimated maximum API cost (USD):", round(estimated_max_cost, 4))
print("API calls made: 0")


Estimated maximum API cost (USD): 0.0586
API calls made: 0


In [17]:
# 输出可以直接复制回聊天框的免费准备摘要
print("=== TWO-LAYER LOCKED PREPARATION SUMMARY FOR REVIEW ===")
print("Notebook version: 08c1-two-layer-locked-preparation-v1")
print("API calls: 0")
print("LLM calls: 0")
print("Frozen rule version:", frozen_rule_manifest["notebook_version"])
print("Locked divisions:", locked_divisions["division_key"].nunique())
print("Locked party queries:", len(locked_queries))
print("Pre-election divisions:", int(locked_divisions["era"].eq("pre_2024_election").sum()))
print("Post-election divisions:", int(locked_divisions["era"].eq("post_2024_election").sum()))
print("Low-confidence object divisions:", int(locked_divisions["extraction_confidence"].eq("low").sum()))
print("Excluded prior-debug divisions:", len(excluded_divisions))
print("Overlap with prior-debug divisions:", len(overlap_with_debug))
print("Policy domains:", locked_divisions["policy_domain_primary"].nunique())
print("Motion families:", locked_divisions["motion_family"].nunique())
print("Mean evidence per query:", round(queries["retrieved_evidence"].mean(), 3))
print("Queries with no evidence:", int(queries["retrieved_evidence"].eq(0).sum()))
print("Fallback model-estimate rows:", int(fallback_estimates["model_estimated_stance"].isin(["support", "oppose"]).sum()))
print("Fallback uncertain rows:", int(fallback_estimates["model_estimated_stance"].eq("insufficient_evidence").sum()))
print("Locked queries SHA-256:", lock_manifest["locked_queries_sha256"])
print("Locked labels SHA-256:", lock_manifest["locked_labels_sha256"])
print("Frozen rule manifest SHA-256:", FROZEN_RULE_MANIFEST_SHA256)
print("Structural gates:", STRUCTURAL_GATES)
print("Estimated maximum API cost (USD):", round(estimated_max_cost, 4))
print("Paid evaluation: NOT RUN")
print("RUN_FINAL_TEST: False")
print("Test result: NOT RUN")
print("Output directory:", OUTPUT_DIR)
print("Next step: review the new lock, fallback distribution and cost before 08c2 paid evaluation.")
print("=== END TWO-LAYER LOCKED PREPARATION SUMMARY ===")


=== TWO-LAYER LOCKED PREPARATION SUMMARY FOR REVIEW ===
Notebook version: 08c1-two-layer-locked-preparation-v1
API calls: 0
LLM calls: 0
Frozen rule version: 08b-instrument-scope-patch-v2.2
Locked divisions: 15
Locked party queries: 60
Pre-election divisions: 10
Post-election divisions: 5
Low-confidence object divisions: 2
Excluded prior-debug divisions: 74
Overlap with prior-debug divisions: 0
Policy domains: 5
Motion families: 2
Mean evidence per query: 4.5
Queries with no evidence: 0
Fallback model-estimate rows: 46
Fallback uncertain rows: 14
Locked queries SHA-256: 90acf73e828b73bc0e3d54794f3a18b1e8fe7614f6434625f9961a021d47c1cc
Locked labels SHA-256: 73a899c20e521bc578661c2a66cfc87d33ce284ad8fc260423ab01026d83c51b
Frozen rule manifest SHA-256: 597c2b2f4ca2869c0ae1c8d79ae05b1b06facc084dbe1777f55ee3de1c2f3a44
Structural gates: {'twenty_divisions_gate': True, 'eighty_queries_gate': True, 'twenty_queries_per_party_gate': True, 'pre_post_balance_gate': True, 'debug_overlap_gate': True